CSolanoM_Proyecto_de_Tecnologías_ll

##Carlos Miguel Solano
###31/08/2026



# 🚗 SISTEMA DE CONTROL DE ACCESO PARA ESTACIONAMIENTO DE MAESTROS
## Documentación Técnica - Proyecto Tecnologías II

### 📌 Contexto del Proyecto
Este sistema tiene como objetivo controlar el acceso al estacionamiento exclusivo para maestros mediante tarjetas NFC. El proyecto se desarrolla como una **maqueta funcional** que demuestra el concepto para una futura implementación a gran escala.

**Componentes físicos utilizados:**
- Arduino (microcontrolador)
- Lector NFC (lectura de tarjetas)
- Sensor ultrasónico (detección de vehículos)
- 2 sensores de presencia (entrada y salida)
- Display LCD (muestra lugares disponibles)
- Motor servo (control de pluma de cartón)
- Protoboard, jumpers y fuente de pilas AA

**Capacidad del estacionamiento:** 10 lugares (configurable)

---

### 👥 Perfiles de Usuarios del Sistema

| Perfil | Rol | Funcionalidades |
|--------|-----|-----------------|
| **Maestro (Usuario Final)** | Persona autorizada con tarjeta NFC | - Ingresar al estacionamiento<br>- Salir del estacionamiento<br>- Visualizar disponibilidad |
| **Administrador del Sistema** | Encargado de la gestión | - Registrar/eliminar tarjetas<br>- Monitorear ocupación<br>- Reiniciar sistema<br>- Ver historial de accesos |
| **Visitante Ocasional** | Persona sin tarjeta permanente | - Acceso temporal (tarjeta de un solo uso)<br>- Registro manual por administrador |

---

### 📊 Estructura de Datos Principales

In [2]:
# ============================================
# ESTRUCTURA DE DATOS DEL SISTEMA
# ============================================

# Configuración inicial
CAPACIDAD_TOTAL = 10  # Número total de lugares en el estacionamiento

# Lista de UIDs autorizados (tarjetas de maestros)
# En la vida real, estos se leerían desde una base de datos o EEPROM
TARJETAS_AUTORIZADAS = [
    "A1B2C3D4",  # Maestro: Juan Pérez
    "E5F6G7H8",  # Maestra: María García
    "I9J0K1L2",  # Maestro: Carlos Rodríguez
    "M3N4O5P6",  # Maestra: Ana Martínez
    "Q7R8S9T0"   # Maestro: Luis Fernández
]

# Diccionario para relacionar UID con nombre del maestro (opcional)
BASE_DATOS_MAESTROS = {
    "A1B2C3D4": "Juan Pérez",
    "E5F6G7H8": "María García",
    "I9J0K1L2": "Carlos Rodríguez",
    "M3N4O5P6": "Ana Martínez",
    "Q7R8S9T0": "Luis Fernández"
}

# Estado del estacionamiento
estado_estacionamiento = {
    "lugares_ocupados": 0,
    "lugares_disponibles": CAPACIDAD_TOTAL,
    "pluma_levantada": False,
    "ultimo_acceso": None,
    "historial_accesos": []  # Lista de diccionarios con eventos
}

# Ejemplo de registro en el historial
# {
#   "uid": "A1B2C3D4",
#   "maestro": "Juan Pérez",
#   "fecha_hora": "2026-09-06 10:30:00",
#   "tipo": "ENTRADA" | "SALIDA",
#   "exitoso": True | False
# }

In [3]:
# ============================================
# FUNCIONES PRINCIPALES DEL SISTEMA
# ============================================

import datetime
import time

# ---------- FUNCIÓN 1: LECTURA DE TARJETA ----------
def leer_tarjeta(uid_simulado=None):
    """
    Simula la lectura de una tarjeta NFC.

    Parámetros:
        uid_simulado (str): UID de la tarjeta (para pruebas). Si es None, simula una lectura aleatoria.

    Retorna:
        str: UID de la tarjeta leída, o None si no se detectó ninguna.

    NOTA: En el hardware real, esta función interactuaría con el lector NFC vía I2C/SPI.
    """
    if uid_simulado:
        return uid_simulado

    # Simulación de lectura: usamos una entrada del usuario para pruebas
    uid = input("🔑 Coloca la tarjeta cerca del lector (simulado): ").strip().upper()
    return uid if uid else None


# ---------- FUNCIÓN 2: VALIDACIÓN DE ACCESO ----------
def validar_acceso(uid):
    """
    Verifica si el UID proporcionado está autorizado para ingresar.

    Parámetros:
        uid (str): UID de la tarjeta a validar.

    Retorna:
        tuple: (bool, str) - (True/False, nombre del maestro o mensaje de error)
    """
    if uid in TARJETAS_AUTORIZADAS:
        nombre = BASE_DATOS_MAESTROS.get(uid, "Maestro desconocido")
        return True, nombre
    else:
        return False, "❌ Acceso denegado: Tarjeta no autorizada"


# ---------- FUNCIÓN 3: CONTAR VEHÍCULOS ----------
def contar_vehiculos():
    """
    Simula la lectura de los sensores de presencia y ultrasónico para
    determinar cuántos lugares están ocupados.

    Retorna:
        int: Número de vehículos actualmente en el estacionamiento.

    NOTA: En hardware real, se usarían los sensores de entrada/salida
    para incrementar/decrementar el contador.
    """
    # En la simulación, retornamos el valor almacenado en el estado
    return estado_estacionamiento["lugares_ocupados"]


# ---------- FUNCIÓN 4: ACTUALIZAR DISPLAY ----------
def actualizar_display():
    """
    Actualiza la información mostrada en el display LCD.
    Muestra lugares disponibles o mensaje de "COMPLETO".

    Retorna:
        str: Mensaje que se mostraría en el display.

    NOTA: En hardware real, se enviaría el mensaje al display vía I2C.
    """
    ocupados = estado_estacionamiento["lugares_ocupados"]
    disponibles = CAPACIDAD_TOTAL - ocupados

    if disponibles > 0:
        mensaje = f"🚗 Lugares disponibles: {disponibles}/{CAPACIDAD_TOTAL}"
    else:
        mensaje = "🚫 ESTACIONAMIENTO COMPLETO"

    print(f"[DISPLAY] {mensaje}")
    return mensaje


# ---------- FUNCIÓN 5: CONTROLAR PLUMA ----------
def controlar_pluma(levantar=True):
    """
    Controla el motor servo para levantar o bajar la pluma de cartón.

    Parámetros:
        levantar (bool): True para levantar la pluma, False para bajarla.

    Retorna:
        str: Estado actual de la pluma.

    NOTA: En hardware real, se enviaría la señal PWM al servo.
    """
    if levantar:
        estado_estacionamiento["pluma_levantada"] = True
        estado = "🟢 PLUMA LEVANTADA - Acceso permitido"
    else:
        estado_estacionamiento["pluma_levantada"] = False
        estado = "🔴 PLUMA BAJADA - Acceso bloqueado"

    print(f"[SERVO] {estado}")
    return estado


# ---------- FUNCIÓN 6: REGISTRAR EVENTO ----------
def registrar_evento(uid, nombre, tipo, exitoso):
    """
    Registra un evento de entrada o salida en el historial del sistema.

    Parámetros:
        uid (str): UID de la tarjeta.
        nombre (str): Nombre del maestro.
        tipo (str): "ENTRADA" o "SALIDA".
        exitoso (bool): True si el acceso fue exitoso.
    """
    evento = {
        "uid": uid,
        "maestro": nombre,
        "fecha_hora": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "tipo": tipo,
        "exitoso": exitoso
    }
    estado_estacionamiento["historial_accesos"].append(evento)
    estado_estacionamiento["ultimo_acceso"] = evento
    print(f"[REGISTRO] {tipo} registrada para {nombre} - {'✅ Exitosa' if exitoso else '❌ Fallida'}")


# ---------- FUNCIÓN 7: PROCESAR ENTRADA ----------
def procesar_entrada(uid):
    """
    Procesa la entrada de un vehículo al estacionamiento.

    Parámetros:
        uid (str): UID de la tarjeta.

    Retorna:
        bool: True si el acceso fue concedido, False en caso contrario.
    """
    # Validar tarjeta
    autorizado, nombre = validar_acceso(uid)

    if not autorizado:
        registrar_evento(uid, "DESCONOCIDO", "ENTRADA", False)
        print(f"⛔ {nombre}")
        return False

    # Verificar disponibilidad
    if estado_estacionamiento["lugares_ocupados"] >= CAPACIDAD_TOTAL:
        print("🚫 No hay lugares disponibles")
        registrar_evento(uid, nombre, "ENTRADA", False)
        return False

    # Conceder acceso
    estado_estacionamiento["lugares_ocupados"] += 1
    estado_estacionamiento["lugares_disponibles"] = CAPACIDAD_TOTAL - estado_estacionamiento["lugares_ocupados"]

    controlar_pluma(levantar=True)
    time.sleep(1)  # Simula tiempo para que el vehículo pase
    controlar_pluma(levantar=False)

    registrar_evento(uid, nombre, "ENTRADA", True)
    actualizar_display()

    print(f"✅ Acceso concedido a {nombre}")
    return True


# ---------- FUNCIÓN 8: PROCESAR SALIDA ----------
def procesar_salida(uid=None):
    """
    Procesa la salida de un vehículo del estacionamiento.
    En la simulación, no requiere UID (se asume que sale un vehículo).

    Parámetros:
        uid (str, opcional): UID de la tarjeta para registro.

    Retorna:
        bool: True si la salida fue procesada correctamente.
    """
    if estado_estacionamiento["lugares_ocupados"] <= 0:
        print("⚠️ No hay vehículos para salir")
        return False

    estado_estacionamiento["lugares_ocupados"] -= 1
    estado_estacionamiento["lugares_disponibles"] = CAPACIDAD_TOTAL - estado_estacionamiento["lugares_ocupados"]

    controlar_pluma(levantar=True)
    time.sleep(1)
    controlar_pluma(levantar=False)

    nombre = "Salida"  # En la simulación no identificamos quién sale
    registrar_evento(uid or "SALIDA", nombre, "SALIDA", True)
    actualizar_display()

    print(f"✅ Salida procesada. Lugares ocupados: {estado_estacionamiento['lugares_ocupados']}")
    return True


# ---------- FUNCIÓN 9: VER HISTORIAL ----------
def ver_historial():
    """
    Muestra el historial completo de accesos del sistema.
    """
    print("\n" + "="*60)
    print("📋 HISTORIAL DE ACCESOS")
    print("="*60)

    if not estado_estacionamiento["historial_accesos"]:
        print("No hay registros aún.")
        return

    for evento in estado_estacionamiento["historial_accesos"]:
        status = "✅" if evento["exitoso"] else "❌"
        print(f"{evento['fecha_hora']} | {evento['tipo']:7} | {evento['maestro']:15} | {status}")

    print("="*60)


# ---------- FUNCIÓN 10: REINICIAR SISTEMA (ADMIN) ----------
def reiniciar_sistema():
    """
    Reinicia el estado del estacionamiento (función administrativa).
    """
    global estado_estacionamiento
    estado_estacionamiento["lugares_ocupados"] = 0
    estado_estacionamiento["lugares_disponibles"] = CAPACIDAD_TOTAL
    estado_estacionamiento["pluma_levantada"] = False
    print("🔄 Sistema reiniciado correctamente")
    actualizar_display()

In [4]:
# ============================================
# DEMOSTRACIÓN DEL SISTEMA
# ============================================

print("="*60)
print("🚗 SISTEMA DE ESTACIONAMIENTO - DEMOSTRACIÓN")
print("="*60)
print(f"Capacidad total: {CAPACIDAD_TOTAL} lugares")
print(f"Tarjetas registradas: {len(TARJETAS_AUTORIZADAS)}")
print("="*60)

# 1. Mostrar estado inicial
actualizar_display()

print("\n" + "-"*40)
print("🔄 SIMULACIÓN DE ENTRADAS")
print("-"*40)

# 2. Simular entradas con tarjetas autorizadas
entradas_exitosas = 0
for uid in TARJETAS_AUTORIZADAS[:3]:  # Probamos con las primeras 3 tarjetas
    print(f"\n--- Intentando entrada con UID: {uid} ---")
    procesar_entrada(uid)
    entradas_exitosas += 1
    time.sleep(0.5)

# 3. Mostrar estado después de entradas
print("\n" + "-"*40)
print("📊 ESTADO ACTUAL")
print("-"*40)
print(f"Lugares ocupados: {estado_estacionamiento['lugares_ocupados']}")
print(f"Lugares disponibles: {estado_estacionamiento['lugares_disponibles']}")

# 4. Intentar entrada con tarjeta no autorizada
print("\n" + "-"*40)
print("🚫 PRUEBA DE TARJETA NO AUTORIZADA")
print("-"*40)
print("\n--- Intentando entrada con UID: Z9Y8X7W6 (no autorizada) ---")
procesar_entrada("Z9Y8X7W6")

# 5. Simular salidas
print("\n" + "-"*40)
print("🔄 SIMULACIÓN DE SALIDAS")
print("-"*40)
for i in range(2):
    print(f"\n--- Procesando salida {i+1} ---")
    procesar_salida()
    time.sleep(0.5)

# 6. Ver historial
ver_historial()

# 7. Estado final
print("\n" + "-"*40)
print("📊 ESTADO FINAL")
print("-"*40)
actualizar_display()

🚗 SISTEMA DE ESTACIONAMIENTO - DEMOSTRACIÓN
Capacidad total: 10 lugares
Tarjetas registradas: 5
[DISPLAY] 🚗 Lugares disponibles: 10/10

----------------------------------------
🔄 SIMULACIÓN DE ENTRADAS
----------------------------------------

--- Intentando entrada con UID: A1B2C3D4 ---
[SERVO] 🟢 PLUMA LEVANTADA - Acceso permitido
[SERVO] 🔴 PLUMA BAJADA - Acceso bloqueado
[REGISTRO] ENTRADA registrada para Juan Pérez - ✅ Exitosa
[DISPLAY] 🚗 Lugares disponibles: 9/10
✅ Acceso concedido a Juan Pérez

--- Intentando entrada con UID: E5F6G7H8 ---
[SERVO] 🟢 PLUMA LEVANTADA - Acceso permitido
[SERVO] 🔴 PLUMA BAJADA - Acceso bloqueado
[REGISTRO] ENTRADA registrada para María García - ✅ Exitosa
[DISPLAY] 🚗 Lugares disponibles: 8/10
✅ Acceso concedido a María García

--- Intentando entrada con UID: I9J0K1L2 ---
[SERVO] 🟢 PLUMA LEVANTADA - Acceso permitido
[SERVO] 🔴 PLUMA BAJADA - Acceso bloqueado
[REGISTRO] ENTRADA registrada para Carlos Rodríguez - ✅ Exitosa
[DISPLAY] 🚗 Lugares disponibles: 7

'🚗 Lugares disponibles: 9/10'

### 🔄 Flujo de Trabajo del Sistema (Diagrama Mermaid)

```mermaid
graph TD
    A[Inicio] --> B[Lector NFC detecta tarjeta]
    B --> C[Leer UID de la tarjeta]
    C --> D{¿UID está autorizado?}
    D -->|No| E[Mostrar Acceso Denegado]
    E --> Z[Fin]
    D -->|Sí| F{¿Hay lugares disponibles?}
    F -->|No| G[Mostrar COMPLETO]
    G --> Z
    F -->|Sí| H[Levantar pluma]
    H --> I[Registrar entrada en historial]
    I --> J[Incrementar contador de ocupados]
    J --> K[Actualizar display]
    K --> L[Esperar paso del vehículo]
    L --> M[Bajar pluma]
    M --> Z


### 📋 Especificaciones Técnicas para Implementación Real

| Componente | Especificación |
|------------|----------------|
| **Arduino** | Arduino Uno o Nano (ATmega328P) |
| **Lector NFC** | PN532 o RC522 (protocolo I2C/SPI) |
| **Sensor ultrasónico** | HC-SR04 (rango: 2cm - 400cm) |
| **Sensores de presencia** | PIR o sensores infrarrojos |
| **Display** | LCD 16x2 con módulo I2C |
| **Servo motor** | SG90 o MG995 (control PWM) |
| **Tarjetas NFC** | Mifare Classic 1K (13.56 MHz) |

### 🔌 Conexiones Sugeridas (Hardware Real)

| Componente | Pin Arduino |
|------------|-------------|
| Lector NFC (SDA) | Pin 10 (SS) |
| Lector NFC (SCK) | Pin 13 |
| Lector NFC (MOSI) | Pin 11 |
| Lector NFC (MISO) | Pin 12 |
| Sensor ultrasónico (TRIG) | Pin 9 |
| Sensor ultrasónico (ECHO) | Pin 8 |
| Sensor presencia entrada | Pin 7 |
| Sensor presencia salida | Pin 6 |
| Display LCD (SDA) | Pin A4 (I2C) |
| Display LCD (SCL) | Pin A5 (I2C) |
| Servo motor (señal) | Pin 3 |

### 🛠️ Posibles Mejoras Futuras

1. **Base de datos externa** para gestionar más tarjetas y maestros.
2. **Registro horario** para controlar accesos en horarios específicos.
3. **Cámara de seguridad** para registrar vehículos.
4. **App móvil** para que los maestros vean disponibilidad en tiempo real.
5. **Panel LED grande** visible desde lejos.
6. **Batería de respaldo** para cortes de energía.

In [5]:
# ============================================
# FUNCIÓN PARA GENERAR REPORTE DEL SISTEMA
# ============================================

def generar_reporte():
    """
    Genera un reporte completo del estado actual del sistema.
    Útil para el administrador.
    """
    print("\n" + "="*60)
    print("📊 REPORTE DEL SISTEMA DE ESTACIONAMIENTO")
    print("="*60)
    print(f"📅 Fecha y hora: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"🏢 Capacidad total: {CAPACIDAD_TOTAL} lugares")
    print(f"🚗 Lugares ocupados: {estado_estacionamiento['lugares_ocupados']}")
    print(f"🟢 Lugares disponibles: {estado_estacionamiento['lugares_disponibles']}")
    print(f"🚧 Pluma: {'🟢 LEVANTADA' if estado_estacionamiento['pluma_levantada'] else '🔴 BAJADA'}")
    print(f"👤 Último acceso: {estado_estacionamiento['ultimo_acceso']}")
    print(f"📋 Total de eventos registrados: {len(estado_estacionamiento['historial_accesos'])}")
    print("="*60)

    # Mostrar resumen de tarjetas activas
    print("\n📇 TARJETAS AUTORIZADAS:")
    for uid, nombre in BASE_DATOS_MAESTROS.items():
        print(f"   🔑 {uid} → {nombre}")
    print("="*60)

# Ejecutar reporte
generar_reporte()


📊 REPORTE DEL SISTEMA DE ESTACIONAMIENTO
📅 Fecha y hora: 2026-09-06 21:03:57
🏢 Capacidad total: 10 lugares
🚗 Lugares ocupados: 1
🟢 Lugares disponibles: 9
🚧 Pluma: 🔴 BAJADA
👤 Último acceso: {'uid': 'SALIDA', 'maestro': 'Salida', 'fecha_hora': '2026-09-06 21:02:23', 'tipo': 'SALIDA', 'exitoso': True}
📋 Total de eventos registrados: 6

📇 TARJETAS AUTORIZADAS:
   🔑 A1B2C3D4 → Juan Pérez
   🔑 E5F6G7H8 → María García
   🔑 I9J0K1L2 → Carlos Rodríguez
   🔑 M3N4O5P6 → Ana Martínez
   🔑 Q7R8S9T0 → Luis Fernández
